In [1]:
import pandas as pd
from pathlib import Path

# Chargeons le fichier qu on vient de nettoyer 

In [2]:
DATA_PATH = Path(
    "../data/processed/primero_bank_clients_nettoyes.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,N° du client,Statut du client,Âge du client,Genre du client,Nb de personnes à charge,Niveau de diplôme,Statut marital,Catégorie du revenu annuel,Type de carte,Durée d'engagement en mois,Nb de mois inactif,Nb d'interactions,Montant crédit renouvellé,Nb de transactions,Utilisation moyenne de la carte
0,768805383,Client actuel,45,M,3,Lycée (équivalent baccalauréat),Marié(e),$60K - $80K,Blue,39,1,3,777,42,0.061
1,818770008,Client actuel,49,F,5,Licence,Célibataire,Moins de $40K,Blue,44,1,2,864,33,0.105
2,713982108,Client actuel,51,M,3,Licence,Marié(e),$80K - $120K,Blue,36,1,0,0,20,0.000
3,769911858,Client actuel,40,F,4,Lycée (équivalent baccalauréat),Non connu,Moins de $40K,Blue,34,4,1,2517,20,0.760
4,709106358,Client actuel,40,M,3,Sans diplôme,Marié(e),$60K - $80K,Blue,21,1,0,0,28,0.000


In [3]:
df.shape

(10127, 15)

In [4]:
df["Statut du client"].value_counts()

Statut du client
Client actuel    8491
Client perdu     1636
Name: count, dtype: int64

# calculons le taux d ' attribution 

In [5]:

nombre_total_clients = len(df)

nombre_clients_perdus = (
    df["Statut du client"] == "Client perdu"
).sum()

taux_attrition = (
    nombre_clients_perdus / nombre_total_clients * 100
)

print("Nombre total de clients :", nombre_total_clients)
print("Nombre de clients perdus :", nombre_clients_perdus)
print(f"Taux d'attrition : {taux_attrition:.2f} %")

Nombre total de clients : 10127
Nombre de clients perdus : 1636
Taux d'attrition : 16.15 %


# Créons une variable binaire d ' attribution 

In [6]:
df["attrition"] = (
    df["Statut du client"] == "Client perdu"
).astype(int)

In [7]:
df[["Statut du client", "attrition"]].head()

,Statut du client,attrition
0,Client actuel,0
1,Client actuel,0
2,Client actuel,0
3,Client actuel,0
4,Client actuel,0


# Comparons les moyennes des clients actuels et perdus 

In [8]:
colonnes_mesures = [
    "Âge du client",
    "Nb de personnes à charge",
    "Durée d'engagement en mois",
    "Nb de mois inactif",
    "Nb d'interactions",
    "Montant crédit renouvellé",
    "Nb de transactions",
    "Utilisation moyenne de la carte"
]

In [9]:
comparaison_statuts = (
    df.groupby("Statut du client")[colonnes_mesures]
    .mean()
    .round(2)
    .T
)

comparaison_statuts

Statut du client,Client actuel,Client perdu
Âge du client,46.26,46.71
Nb de personnes à charge,2.34,2.40
Durée d'engagement en mois,35.88,36.18
Nb de mois inactif,2.27,3.23
Nb d'interactions,2.36,3.48
Montant crédit renouvellé,1256.10,678.65
Nb de transactions,68.65,45.18
Utilisation moyenne de la carte,0.30,0.16


# Calculons l'ecart entre client actuel et perdu 

In [10]:
comparaison_statuts["Écart"] = (
    comparaison_statuts["Client perdu"]
    - comparaison_statuts["Client actuel"]
)

comparaison_statuts.sort_values(
    by="Écart",
    key=lambda x: x.abs(),
    ascending=False
)

Statut du client,Client actuel,Client perdu,Écart
Montant crédit renouvellé,1256.10,678.65,-577.45
Nb de transactions,68.65,45.18,-23.47
Nb d'interactions,2.36,3.48,1.12
Nb de mois inactif,2.27,3.23,0.96
Âge du client,46.26,46.71,0.45
Durée d'engagement en mois,35.88,36.18,0.30
Utilisation moyenne de la carte,0.30,0.16,-0.14
Nb de personnes à charge,2.34,2.40,0.06


# Analysons les variables catégorielles 

In [11]:
def taux_attrition_par_categorie(data, colonne):
    resultat = (
        data.groupby(colonne)["attrition"]
        .agg(
            nombre_clients="count",
            clients_perdus="sum",
            taux_attrition="mean"
        )
        .reset_index()
    )

    resultat["taux_attrition"] = (
        resultat["taux_attrition"] * 100
    ).round(2)

    return resultat.sort_values(
        "taux_attrition",
        ascending=False
    )

In [12]:
taux_attrition_par_categorie(
    df,
    "Genre du client"
)

,Genre du client,nombre_clients,clients_perdus,taux_attrition
0,F,5358,932,17.39
1,M,4769,704,14.76


In [13]:
taux_attrition_par_categorie(
    df,
    "Niveau de diplôme"
)

,Niveau de diplôme,nombre_clients,clients_perdus,taux_attrition
0,Doctorat,451,96,21.29
3,Master,516,94,18.22
5,Non connu,1519,256,16.85
6,Sans diplôme,1487,238,16.01
1,Licence,3128,492,15.73
2,Lycée (équivalent baccalauréat),2013,306,15.20
4,Niveau Bac+2,1013,154,15.20


In [14]:
taux_attrition_par_categorie(
    df,
    "Statut marital"
)

,Statut marital,nombre_clients,clients_perdus,taux_attrition
2,Marié(e),4547,937,20.61
3,Non connu,749,130,17.36
1,Divorcé(e),748,122,16.31
0,Célibataire,4083,447,10.95


In [15]:
taux_attrition_par_categorie(
    df,
    "Catégorie du revenu annuel"
)

,Catégorie du revenu annuel,nombre_clients,clients_perdus,taux_attrition
2,$60K - $80K,1577,367,23.27
1,$40K - $60K,1961,442,22.54
0,$120K +,727,129,17.74
3,$80K - $120K,1569,277,17.65
5,Non connu,1110,187,16.85
4,Moins de $40K,3183,234,7.35


In [16]:
taux_attrition_par_categorie(
    df,
    "Type de carte"
)

,Type de carte,nombre_clients,clients_perdus,taux_attrition
2,Platinum,20,14,70.00
1,Gold,116,21,18.10
0,Blue,9436,1519,16.10
3,Silver,555,82,14.77


In [17]:
taux_attrition_par_categorie(
    df,
    "Nb de mois inactif"
)

,Nb de mois inactif,nombre_clients,clients_perdus,taux_attrition
7,7,54,54,100.00
8,8,19,19,100.00
6,6,269,164,60.97
0,0,29,15,51.72
5,5,282,136,48.23
4,4,369,65,17.62
3,3,3590,572,15.93
2,2,3282,509,15.51
1,1,2233,102,4.57


# Analysons le nombre de transactions 

In [18]:
df["groupe_transactions"] = pd.cut(
    df["Nb de transactions"],
    bins=[-1, 20, 40, 60, 80, float("inf")],
    labels=[
        "0-20",
        "21-40",
        "41-60",
        "61-80",
        "81 et plus"
    ]
)

In [19]:
taux_attrition_par_categorie(
    df,
    "groupe_transactions"
)

,groupe_transactions,nombre_clients,clients_perdus,taux_attrition
0,0-20,119,81,68.07
2,41-60,2109,780,36.98
1,21-40,1819,532,29.25
3,61-80,3530,205,5.81
4,81 et plus,2550,38,1.49


# Analysons l'age par groupe 

In [20]:
df["groupe_age"] = pd.cut(
    df["Âge du client"],
    bins=[0, 30, 40, 50, 60, 100],
    labels=[
        "Moins de 30 ans",
        "30-39 ans",
        "40-49 ans",
        "50-59 ans",
        "60 ans et plus"
    ],
    right=False
)

In [21]:
taux_attrition_par_categorie(
    df,
    "groupe_age"
)

,groupe_age,nombre_clients,clients_perdus,taux_attrition
3,50-59 ans,3001,511,17.03
2,40-49 ans,4555,773,16.97
1,30-39 ans,1841,261,14.18
4,60 ans et plus,535,74,13.83
0,Moins de 30 ans,195,17,8.72


## Analyse synthétique des premiers résultats

Les résultats montrent que l’attrition n’est pas liée à un seul facteur, mais à un ensemble de signaux comportementaux et démographiques qui se rejoignent. Les clients perdus apparaissent globalement plus faibles sur plusieurs dimensions d’engagement bancaire.

### Signaux les plus visibles

- un niveau d’inactivité plus élevé, mesuré par un nombre de mois sans activité plus important ;
- moins de transactions enregistrées sur la période observée ;
- une utilisation moyenne de la carte plus limitée ;
- moins d’interactions avec la banque, ce qui peut traduire un lien plus faible avec le produit ;
- une concentration dans certains segments de clientèle, notamment selon l’âge, le revenu ou le type de carte.

### Interprétation utile

Dans l’ensemble, on observe un profil de client potentiellement à risque : moins actif, moins engagé dans ses usages bancaires et moins connecté à la relation avec la banque. Ces éléments sont cohérents avec une probabilité plus élevée de départ.

### Limite à garder en tête

Il s’agit ici d’associations observées dans les données, pas d’une preuve de causalité. Les facteurs identifiés doivent donc être considérés comme des pistes prioritaires pour la mise en place d’un modèle de scoring et d’actions de fidélisation ciblées.
